# Fine-tuning RuBERT для классификации тональности отзывов

Датасет: [angryelizar/sentiment_dataset_splitted](https://huggingface.co/datasets/angryelizar/sentiment_dataset_splitted)  
Модель: [ai-forever/ruBert-base](https://huggingface.co/ai-forever/ruBert-base)

Задача: классифицировать отзывы на `neutral`, `positive`, `negative`.

*Подключите графический ускоритель*

## Поготовка

In [ ]:
%pip install -q --upgrade \
    "transformers==5.22.2" \
    "datasets==4.8.5" \
    "huggingface-hub>=1.32,<3" \
    "pyarrow==23.0.1" \
    evaluate \
    accelerate \
    scikit-learn

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Проверяем устройство

In [ ]:
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Память GPU:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
print("BF16:", USE_BF16, "| FP16:", USE_FP16)

PyTorch: 2.14.1+cpu
BF16: False | FP16: True


Подключаем Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/rubert_sentiment"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "final_model")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print("Папка проекта:", PROJECT_DIR)

Mounted at /content/drive
Папка проекта: /content/drive/MyDrive/rubert_sentiment


## Загрузка датасета

In [ ]:
DATASET_NAME = "angryelizar/sentiment_dataset_splitted"
dataset = load_dataset(DATASET_NAME)

README.md:   0%|          | 0.00/6.63k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 78.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.5MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/232366 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/58092 [00:00<?, ? examples/s]

Кратко осмотрим датасет

In [ ]:
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 232366
    })
    test: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 58092
    })
})
{'text': Value('string'), 'label': ClassLabel(names=['neutral', 'positive', 'negative']), 'src': Value('string')}


Проверим сбалансированность

In [ ]:
label_names = dataset["train"].features["label"].names
for split_name in dataset:
    print(f"{split_name}:")
    counts = (
        pd.Series(dataset[split_name]["label"])
        .value_counts()
        .sort_index()
    )
    for label, count in counts.items():
        print(f"\t{label_names[label]}: {count}")

train:
	neutral: 77271
	positive: 77502
	negative: 77593
test:
	neutral: 19318
	positive: 19375
	negative: 19399


Разделим `train` на две части: 90% для обучения и 10% для валидации

In [ ]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="label",
)

train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 209129
Validation: 23237
Test: 58092


## Токенизация (2)

`MAX_LENGTH` был определен отдельно: оптимальные значения 512 или 382

In [ ]:
MODEL_NAME = "ai-forever/ruBert-base"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

Токенезируем датасет

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text", "src"],
    desc="Tokenizing train",
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text", "src"],
    desc="Tokenizing validation",
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text", "src"],
    desc="Tokenizing test",
)

Tokenizing train:   0%|          | 0/209129 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/23237 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/58092 [00:00<?, ? examples/s]

In [ ]:
print("Столбцы:", tokenized_train.column_names)
print("Train:", len(tokenized_train))
print("Validation:", len(tokenized_valid))
print("Test:", len(tokenized_test))

Столбцы: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Train: 209129
Validation: 23237
Test: 58092


In [ ]:
print("Пример токенизированных данных:")
print(tokenized_train[0])

Пример токенизированных данных:
{'label': 1, 'input_ids': [101, 32326, 378, 41080, 113, 1303, 660, 42307, 750, 107, 654, 33963, 5404, 780, 2184, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


Подготовим collator, который будет выравнивать длину отзывов в батчах

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Загрузка RuBERT

Загружаем предобученный RuBERT и добавляем классификационный слой, который будет определять тональность отзыва

In [ ]:
id2label = {
    0: "neutral",
    1: "positive",
    2: "negative",
}
label2id = {name: idx for idx, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

print("Модель:", MODEL_NAME)
print("Классов:", model.config.num_labels)
print("Параметров:", sum(p.numel() for p in model.parameters()))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  716MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  716MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params we

Модель: ai-forever/ruBert-base
Классов: 3
Параметров: 178309635


## Дообучение модели (3)

Настроим метрики

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

Настроим fine-tuning

In [ ]:
import math

BATCH_SIZE = 4
GRAD_ACCUMULATION = 4

TOTAL_STEPS = math.ceil(
    len(tokenized_train) / (BATCH_SIZE * GRAD_ACCUMULATION)
)

WARMUP_STEPS = int(TOTAL_STEPS * 0.06)

print("Шагов за эпоху:", TOTAL_STEPS)
print("Шагов прогрева:", WARMUP_STEPS)

Шагов за эпоху: 13071
Шагов прогрева: 784


In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Обучение
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUMULATION,

    # Оптимизация
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    optim="adamw_torch",

    # Вычисления
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=False,

    # Валидация
    eval_strategy="steps",
    eval_steps=1500,

    # Сохранение моделей
    save_strategy="steps",
    save_steps=1500,
    save_total_limit=1,

    # Выбор лучшей модели
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,

    # Логирование
    logging_steps=100,
    report_to="none",

    # Воспроизводимость
    seed=SEED,
    data_seed=SEED,

    # Загрузка данных
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    eval_accumulation_steps=16,
)

Создадим Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Дообучим модель

In [ ]:
if os.path.exists(os.path.join(FINAL_MODEL_DIR, "config.json")):
    print("Обнаружена сохраненная финальная модель. Загружаем веса...")
    model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
    trainer.model = model.to(training_args.device)
else:
    last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
    print("Последний чекпоинт:", last_checkpoint or "не найден — начинаем с начала")

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    print(train_result)

    trainer.save_model(FINAL_MODEL_DIR)
    print("Модель сохранена в:", FINAL_MODEL_DIR)

Обнаружена сохраненная финальная модель. Загружаем веса...


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

## Оценка на тестовой выборке

In [ ]:
metrics_path = os.path.join(PROJECT_DIR, "test_metrics.json")

if os.path.exists(metrics_path):
    print("Обнаружены сохраненные результаты тестов на диске. Загружаем...")
    with open(metrics_path, "r", encoding="utf-8") as f:
        saved_metrics = json.load(f)
    print("\nTest metrics:")
    print(json.dumps(saved_metrics, indent=2))
else:
    print("Результаты не найдены на диске. Запускаем предсказание модели...")
    test_output = trainer.predict(tokenized_test, metric_key_prefix="test")
    test_predictions = np.argmax(test_output.predictions, axis=-1)

    print("Test metrics:")
    metrics_to_save = {
        "accuracy": float(accuracy_score(test_output.label_ids, test_predictions)),
        "f1_macro": float(f1_score(
            test_output.label_ids,
            test_predictions,
            average="macro",
        )),
    }
    print(json.dumps(metrics_to_save, indent=2))

    print("\nClassification report:")
    print(classification_report(
        test_output.label_ids,
        test_predictions,
        target_names=["neutral", "positive", "negative"],
        digits=4,
    ))

    print("Confusion matrix:")
    print(confusion_matrix(test_output.label_ids, test_predictions))

    print("\nСохраняем токенайзер и результаты новых тестов...")
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_to_save, f, ensure_ascii=False, indent=2)
    print("Результаты сохранены в:", metrics_path)

Обнаружены сохраненные результаты тестов на диске. Загружаем...

Test metrics:
{
  "accuracy": 0.7767850994973491,
  "f1_macro": 0.7769202922485535
}


## Тесты на произвольных текстах

In [ ]:
import torch.nn.functional as F

custom_texts = [
    "Очень вкусные булочки",

    # Маленькие отзывы
    "Доставка приехала быстро, товар отличного качества.",
    "Ужасный сервис, больше никогда здесь не закажу.",
    "Обычный магазин, ничего особенного я не заметил.",

    # Большие отзывы
    "Заказала этот диван на прошлой неделе. Качество обивки просто великолепное, все швы ровные.",
    "Телефон постоянно зависает при просмотре видео. Полное разочарование, оформляю возврат средств.",
    "Приложение работает стабильно после последнего обновления. Однако некоторые старые функции куда-то пропали.",

    # С отрицаниями и путаницей (противоречивые / сарказм)
    "Не могу сказать, что фильм ужасный, но и хорошего мало.",
    "Ожидал полного провала, однако качество оказалось совсем не таким плохим.",
    "Безумно 'благодарен' за испорченный вечер и холодную пиццу."
]

inputs = tokenizer(custom_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    # Softmax
    probabilities = F.softmax(outputs.logits, dim=-1)
    predictions = torch.argmax(outputs.logits, dim=-1)

print(f"{'Распределение (Neu / Pos / Neg)':<33} | {'Прогноз':<9} | Текст")
for text, pred_id, probs in zip(custom_texts, predictions, probabilities):
    label = id2label[pred_id.item()]
    probs_str = f"Neu: {probs[0]:.2f} | Pos: {probs[1]:.2f} | Neg: {probs[2]:.2f}"
    print(f"{probs_str:<32} | {label:<9} | {text}")

Распределение (Neu / Pos / Neg)   | Прогноз   | Текст
Neu: 0.02 | Pos: 0.96 | Neg: 0.01 | positive  | Очень вкусные булочки
Neu: 0.01 | Pos: 0.98 | Neg: 0.01 | positive  | Доставка приехала быстро, товар отличного качества.
Neu: 0.01 | Pos: 0.00 | Neg: 0.99 | negative  | Ужасный сервис, больше никогда здесь не закажу.
Neu: 0.96 | Pos: 0.02 | Neg: 0.02 | neutral   | Обычный магазин, ничего особенного я не заметил.
Neu: 0.01 | Pos: 0.99 | Neg: 0.00 | positive  | Заказала этот диван на прошлой неделе. Качество обивки просто великолепное, все швы ровные.
Neu: 0.14 | Pos: 0.01 | Neg: 0.85 | negative  | Телефон постоянно зависает при просмотре видео. Полное разочарование, оформляю возврат средств.
Neu: 0.56 | Pos: 0.39 | Neg: 0.06 | neutral   | Приложение работает стабильно после последнего обновления. Однако некоторые старые функции куда-то пропали.
Neu: 0.55 | Pos: 0.03 | Neg: 0.42 | neutral   | Не могу сказать, что фильм ужасный, но и хорошего мало.
Neu: 0.79 | Pos: 0.04 | Neg: 0.17 | neu